CREATING THE BASELINE MODEL
- lowercase text -> TF-IDF -> logistic regression

In [ ]:
# Loading the splits from data/processed directory
from pathlib import Path
import pandas as pd

processed_dir = Path("../data/processed")
train_df = pd.read_csv(processed_dir / "train.csv", keep_default_na=False)
validation_df = pd.read_csv(processed_dir / "validation.csv", keep_default_na=False)

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(validation_df)}")

Training rows: 3617
Validation rows: 775


In [ ]:
# Converting the lables to numeric values 
label_mapping = {"ham": 0, "spam": 1}

train_df["label"] = train_df["label"].map(label_mapping)
validation_df["label"] = validation_df["label"].map(label_mapping)

assert train_df["label"].notna().all()
assert validation_df["label"].notna().all()
assert set(train_df["label"].unique()) <= {0, 1}
assert set(validation_df["label"].unique()) <= {0, 1}

print("Label mapping: ham = 0, spam = 1")
print("Training class counts:")
print(train_df["label"].value_counts().sort_index())
print("Validation class counts:")
print(validation_df["label"].value_counts().sort_index())

Label mapping: ham = 0, spam = 1
Training class counts:
label
0    3167
1     450
Name: count, dtype: int64
Validation class counts:
label
0    679
1     96
Name: count, dtype: int64


SMS text
   ↓
TF-IDF vectorizer
   ↓
Logistic regression
   ↓
ham/spam prediction and probabilities

In [8]:
# Building the pipelines for unigrams and unigrams + bigrams
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline

X_train = train_df["text"]
y_train = train_df["label"]
X_validation = validation_df["text"]
y_validation = validation_df["label"]

pipeline_settings = {
    "lowercase": True,
    "min_df": 2,
    "max_iter": 1000,
}

unigram_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=pipeline_settings["lowercase"],
        ngram_range=(1, 1),
        min_df=pipeline_settings["min_df"],
    )),
    ("logistic_regression", LogisticRegression(
        max_iter=pipeline_settings["max_iter"],
        random_state=42,
    )),
])

bigram_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=pipeline_settings["lowercase"],
        ngram_range=(1, 2),
        min_df=pipeline_settings["min_df"],
    )),
    ("logistic_regression", LogisticRegression(
        max_iter=pipeline_settings["max_iter"],
        random_state=42,
    )),
])

results = []
validation_predictions = {}
validation_probabilities = {}
for name, pipeline in {
    "unigrams": unigram_pipeline,
    "unigrams_and_bigrams": bigram_pipeline,
}.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_validation)
    probabilities = pipeline.predict_proba(X_validation)

    validation_predictions[name] = predictions
    validation_probabilities[name] = probabilities

    assert predictions.shape == (len(X_validation),)
    assert probabilities.shape == (len(X_validation), 2)
    assert (probabilities[:, 1] >= 0).all()
    assert (probabilities[:, 1] <= 1).all()

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_validation, predictions),
        "spam_f1": f1_score(y_validation, predictions, pos_label=1),
    })

results_df = pd.DataFrame(results).sort_values("spam_f1", ascending=False)
print(results_df.to_string(index=False))
print("\nColumn 1 in predict_proba is the spam probability.")

               model  accuracy  spam_f1
            unigrams  0.967742 0.853801
unigrams_and_bigrams  0.963871 0.829268

Column 1 in predict_proba is the spam probability.


In [ ]:
# Measuring the performance of the models on the validation set
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)

validation_metrics = {}
precision_recall_data = {}

for name in validation_predictions:
    predictions = validation_predictions[name]
    spam_probabilities = validation_probabilities[name][:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_validation,
        spam_probabilities,
        pos_label=1,
    )
    average_precision = average_precision_score(
        y_validation,
        spam_probabilities,
        pos_label=1,
    )

    validation_metrics[name] = {
        "confusion_matrix": confusion_matrix(y_validation, predictions),
        "classification_report": classification_report(
            y_validation,
            predictions,
            target_names=["ham", "spam"],
            output_dict=True,
        ),
        "average_precision": average_precision,
    }
    precision_recall_data[name] = {
        "precision": precision,
        "recall": recall,
        "thresholds": thresholds,
    }

    print(f"\n{name}")
    print("Confusion matrix [rows=true, columns=predicted]:")
    print(validation_metrics[name]["confusion_matrix"])
    print("Classification report:")
    print(classification_report(
        y_validation,
        predictions,
        target_names=["ham", "spam"],
    ))
    print(f"Average precision: {average_precision:.4f}")


unigrams
Confusion matrix [rows=true, columns=predicted]:
[[677   2]
 [ 23  73]]
Classification report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       679
        spam       0.97      0.76      0.85        96

    accuracy                           0.97       775
   macro avg       0.97      0.88      0.92       775
weighted avg       0.97      0.97      0.97       775

Average precision: 0.9746

unigrams_and_bigrams
Confusion matrix [rows=true, columns=predicted]:
[[679   0]
 [ 28  68]]
Classification report:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       679
        spam       1.00      0.71      0.83        96

    accuracy                           0.96       775
   macro avg       0.98      0.85      0.90       775
weighted avg       0.97      0.96      0.96       775

Average precision: 0.9798


## Understanding the validation metrics

### False positives and false negatives

For this spam detector:

- False positive: A ham message is predicted as spam. This could cause a legitimate message to be incorrectly blocked or sent to a spam folder.
- False negative: A spam message is predicted as ham. This is a missed spam message, so the unwanted or potentially malicious message reaches the user.

### Why accuracy alone is misleading

The validation set contains 679 ham messages and 96 spam messages. If the model always predicted ham, it would be correct for all 679 ham messages and wrong for all 96 spam messages:

```text
679 / 775 = about 87.6% accuracy
```

That sounds good, but the model would detect **zero spam messages**. Its spam recall would be 0, making it useless for the actual goal of detecting spam.

### Spam precision and recall

For the unigram model on the validation set:

- Spam precision = 0.97: Of every 100 messages the model flags as spam, about 97 are truly spam and about 3 are actually ham.
- Spam recall = 0.76: Of every 100 spam messages, the model detects about 76 and misses about 24.

Spam is the positive class because it is the type of message we want to detect.

In [18]:
# Inspect model weights and validation errors for the selected unigram model
selected_pipeline = unigram_pipeline
vectorizer = selected_pipeline.named_steps["tfidf"]
classifier = selected_pipeline.named_steps["logistic_regression"]

vocabulary = vectorizer.get_feature_names_out()
weights = classifier.coef_[0]
feature_weights = pd.DataFrame({
    "word": vocabulary,
    "weight": weights,
})

print("Top 20 words toward spam:")
print(feature_weights.nlargest(20, "weight").to_string(index=False))
print("\nTop 20 words toward ham:")
print(feature_weights.nsmallest(20, "weight").sort_values("weight", ascending=True).to_string(index=False))

error_analysis = validation_df.reset_index(drop=True).copy()
error_analysis["predicted_label"] = validation_predictions["unigrams"]
error_analysis["spam_probability"] = validation_probabilities["unigrams"][:, 1]

false_positives = error_analysis[
    (error_analysis["label"] == 0) & (error_analysis["predicted_label"] == 1)
].sort_values("spam_probability", ascending=False)
false_negatives = error_analysis[
    (error_analysis["label"] == 1) & (error_analysis["predicted_label"] == 0)
].sort_values("spam_probability", ascending=True)

print(f"\nFalse positives: showing {min(10, len(false_positives))} of {len(false_positives)}")
print(false_positives.head(10)[["label", "predicted_label", "spam_probability", "text"]].to_string(index=False))
print("\nFalse negatives: showing up to 10")
print(false_negatives.head(10)[["label", "predicted_label", "spam_probability", "text"]].to_string(index=False))

Top 20 words toward spam:
   word   weight
   call 4.050029
    txt 3.669151
   free 3.546685
   text 2.941018
   stop 2.771174
     uk 2.732901
    www 2.637235
     to 2.613564
 mobile 2.588974
  claim 2.494045
   150p 2.461951
  reply 2.415466
  prize 2.273905
   your 2.186457
service 2.155722
     50 2.136869
    won 2.125842
    com 2.053162
   from 2.035284
    win 1.979815

Top 20 words toward ham:
 word    weight
   me -2.353899
   my -1.949415
 that -1.388575
  can -1.384962
  but -1.370527
   gt -1.340847
   ok -1.340501
   lt -1.330202
 when -1.314323
  how -1.289880
   da -1.238043
   am -1.195897
   he -1.116909
going -1.115183
   it -1.112163
 come -1.102981
  got -1.097042
 home -1.090139
   ll -1.076530
   at -1.017163

False positives: showing 2 of 2
 label  predicted_label  spam_probability                                                                                                                                                         text
     0                1